In [ ]:
!pip install pyspark

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, to_date, year

In [ ]:
spark = SparkSession.builder.getOrCreate()

In [ ]:
# 1 Leia o arquivo 'videos-stats.csv' no dataframe 'df_video' com cabeçalho e inferindo o esquema
df_video = spark.read.option("multiLine", "true").option("escape", '"').csv("videos-stats.csv", header=True, inferSchema=True)

In [ ]:
# 2 Altere os valores nulos dos campos 'Likes', 'Comments' e 'Views' para o valor 0
df_video = df_video.fillna({'Likes': 0, 'Comments': 0, 'Views': 0})

In [ ]:
# 3 Leia o arquivo 'comments.csv' no dataframe 'df_comentario' com cabeçalho e inferindo o esquema
df_comentario = spark.read.option("multiLine", "true").option("escape", '"').csv("comments.csv", header=True, inferSchema=True)

In [ ]:
# 4 Calcule a quantidade de registros do df_video e df_comentario
print(f"Quantidade de registros em df_video: {df_video.count()}")
print(f"Quantidade de registros em df_comentario: {df_comentario.count()}")

Quantidade de registros em df_video: 1881
Quantidade de registros em df_comentario: 18409


In [ ]:
# 5 Remova os registros do df_video e df_comentario quem possuem o campo 'Video ID' nulos e calcule novamente a quantidade de registros
df_video = df_video.dropna(subset=["Video ID"])
df_comentario = df_comentario.dropna(subset=["Video ID"])

print(f"Após remoção de nulos - df_video: {df_video.count()}")
print(f"Após remoção de nulos - df_comentario: {df_comentario.count()}")

Após remoção de nulos - df_video: 1881
Após remoção de nulos - df_comentario: 18409


In [ ]:
# 6 Remova os registros apenas do df_video que possuem o campo 'Video ID' duplicados
df_video = df_video.dropDuplicates(["Video ID"])
# Otimização de performance sugerida pelo tutor
df_video = df_video.cache()

In [ ]:
# 7 Converta os campos Likes, Comments e Views para 'int' no dataframe df_video
df_video = df_video.withColumn("Likes", col("Likes").cast("long")) \
                   .withColumn("Comments", col("Comments").cast("long")) \
                   .withColumn("Views", col("Views").cast("long"))

In [ ]:
# 8 Converta os campos Likes e Sentiment para 'int' no dataframe df_comentario, além disso, altere o nome do campo Likes para 'Likes Comment'
df_comentario = df_comentario.withColumnRenamed("Likes", "Likes Comment") \
                             .withColumn("Likes Comment", col("Likes Comment").cast("int")) \
                             .withColumn("Sentiment", col("Sentiment").cast("int"))

In [ ]:
# 9 Crie o campo 'Interaction' no dataframe df_video, com a soma dos campos Likes, Comments e Views
df_video = df_video.withColumn("Interaction", col("Likes") + col("Comments") + col("Views"))

In [ ]:
# 10 Converta os campos 'Published At' para 'date' no dataframe df_video
df_video = df_video.withColumn("Published At", to_date(col("Published At")))

In [ ]:
# 11 Crie o campo 'Year' no dataframe df_video, extraindo apenas o ano do campo 'Published At'
df_video = df_video.withColumn("Year", year(col("Published At")))

In [ ]:
# 12 Mescle os dados df_comentario no dataframe df_video em relação ao campo Video ID e crie o dataframe df_join_video_comments
df_join_video_comments = df_video.join(df_comentario, on="Video ID", how="inner")
df_join_video_comments.show(5)

+-----------+---+--------------------+------------+-------+-----+--------+------+-----------+----+---+--------------------+-------------+---------+
|   Video ID|_c0|               Title|Published At|Keyword|Likes|Comments| Views|Interaction|Year|_c0|             Comment|Likes Comment|Sentiment|
+-----------+---+--------------------+------------+-------+-----+--------+------+-----------+----+---+--------------------+-------------+---------+
|wAZZ-UWGVHI|  0|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672|135612|     139691|2022|  0|Let's not forget ...|           95|        1|
|wAZZ-UWGVHI|  0|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672|135612|     139691|2022|  1|Here in NZ 50% of...|           19|        0|
|wAZZ-UWGVHI|  0|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672|135612|     139691|2022|  2|I will forever ac...|          161|        2|
|wAZZ-UWGVHI|  0|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672|135612|     139691|2022|  3|Whenever I

In [ ]:
# 13 Leia o arquivo 'USvideos.csv' no dataframe 'df_us_videos' com cabeçalho e inferindo o esquema
df_us_videos = spark.read.csv("USvideos.csv", header=True, inferSchema=True)

In [ ]:
# 14 Mescle os dados df_us_videos no dataframe df_video em relação ao campo Title e crie e visualize o dataframe df_join_video_usvideos
df_join_video_usvideos = df_video.join(df_us_videos, on="Title", how="inner")
df_join_video_usvideos.show(5)

+--------------------+---+-----------+------------+-------+------+--------+---------+-----------+----+-----------+-------------+-------------+-----------+--------------------+--------------------+--------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|               Title|_c0|   Video ID|Published At|Keyword| Likes|Comments|    Views|Interaction|Year|   video_id|trending_date|channel_title|category_id|        publish_time|                tags|   views| likes|dislikes|comment_count|      thumbnail_link|comments_disabled|ratings_disabled|video_error_or_removed|         description|
+--------------------+---+-----------+------------+-------+------+--------+---------+-----------+----+-----------+-------------+-------------+-----------+--------------------+--------------------+--------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------

In [ ]:
# 15 Verifique a quantidade de campos nulos em todos os campos do dataframe df_video
from pyspark.sql.functions import sum as _sum
df_video.select([_sum(col(c).isNull().cast("int")).alias(c) for c in df_video.columns]).show()

+---+-----+--------+------------+-------+-----+--------+-----+-----------+----+
|_c0|Title|Video ID|Published At|Keyword|Likes|Comments|Views|Interaction|Year|
+---+-----+--------+------------+-------+-----+--------+-----+-----------+----+
|  0|    0|       0|           0|      0|    0|       0|    0|          0|   0|
+---+-----+--------+------------+-------+-----+--------+-----+-----------+----+



In [ ]:
# 16 Remova a coluna '_c0' e salve o dataframe df_video como 'videos-tratados-parquet' no formato parquet e adicione o cabeçalho nos dados
if "_c0" in df_video.columns:
    df_video_final = df_video.drop("_c0")
else:
    df_video_final = df_video

df_video_final.write.mode("overwrite").parquet("videos-tratados-parquet")

In [ ]:
# 17 Remova a coluna '_c0' e salve o dataframe df_join_video_comments como 'videos-comments-tratados-parquet' no formato parquet e adicione o cabeçalho nos dados
if "_c0" in df_join_video_comments.columns:
    df_join_video_comments_final = df_join_video_comments.drop("_c0")
else:
    df_join_video_comments_final = df_join_video_comments

df_join_video_comments_final.write.mode("overwrite").parquet("videos-comments-tratados-parquet")